In [1]:
import folium
import polars as pl
import mcr_py

In [2]:
bounding_box = [
    (6.912789559592028, 50.95082420629814),
    (6.90965014627875, 50.94788724437913),
    (6.912218504273028, 50.94473937866948),
    (6.914029522915655, 50.944306400295005),
    (6.916940125566271, 50.946735585299706),
    (6.919886618643858, 50.94555227853371),
    (6.9237266503584465, 50.94991721673625),
    (6.926592441418052, 50.951705692912014),
    (6.925031390065072, 50.95226859486215),
    (6.923008239592917, 50.95206669530157),
    (6.923878775814018, 50.952914472192305),
    (6.921391875103581, 50.95386602541586),
    (6.919528164440663, 50.951875256766186),
    (6.912789559592028, 50.95082420629814),
]

mcr_py.load_osm_walking(
    "koeln", bounding_box, "../osmtools/data/", "../osmtools/data", False
)
mcr_py.load_osm_cycling(
    "koeln", bounding_box, True, "../osmtools/data/", "../osmtools/data", False
)
mcr_py.load_osm_driving(
    "koeln", bounding_box, "../osmtools/data/", "../osmtools/data", False
)
mcr_py.load_osm_pois(
    "koeln",
    bounding_box,
    "../osmtools/data/",
    "../osmtools/data/koeln_walking_nodes.csv",
    "../osmtools/data",
    False,
)

In [3]:
df_nodes = pl.read_csv("../osmtools/data/koeln_walking_nodes.csv")
df_c_nodes = pl.read_csv("../osmtools/data/koeln_cycling_nodes.csv")
df_d_nodes = pl.read_csv("../osmtools/data/koeln_driving_nodes.csv")
df_edges = pl.read_csv("../osmtools/data/koeln_walking_edges.csv")
df_c_edges = pl.read_csv("../osmtools/data/koeln_cycling_edges.csv")
df_d_edges = pl.read_csv("../osmtools/data/koeln_driving_edges.csv")
df_pois = pl.read_csv("../osmtools/data/koeln_pois_nodes.csv")
df_pois = df_pois.join(
    df_nodes.select(
        pl.col("osm_id"),
        pl.col("lat").alias("nearest_lat"),
        pl.col("long").alias("nearest_lon"),
    ),
    left_on="nearest_osm_node",
    right_on="osm_id",
)


def get_coordinates(df_edges, df_nodes):
    return df_edges.join(
        df_nodes.select(
            pl.col("osm_id"),
            pl.col("lat").alias("from_lat"),
            pl.col("long").alias("from_lon"),
        ),
        left_on="source_osm",
        right_on="osm_id",
    ).join(
        df_nodes.select(
            pl.col("osm_id"),
            pl.col("lat").alias("dest_lat"),
            pl.col("long").alias("dest_lon"),
        ),
        left_on="dest_osm",
        right_on="osm_id",
    )


df_edges = get_coordinates(df_edges, df_nodes)
df_c_edges = get_coordinates(df_c_edges, df_c_nodes)
df_d_edges = get_coordinates(df_d_edges, df_d_nodes)

In [4]:
df_edges = df_edges.with_columns(
    pl.concat_list(pl.col("source").cast(pl.String), pl.col("dest").cast(pl.String))
    .list.sort()
    .list.join("")
    .is_duplicated()
    .alias("duplicated")
)
df_c_edges = df_c_edges.with_columns(
    pl.concat_list(pl.col("source").cast(pl.String), pl.col("dest").cast(pl.String))
    .list.sort()
    .list.join("")
    .is_duplicated()
    .alias("duplicated")
)
df_d_edges = df_d_edges.with_columns(
    pl.concat_list(pl.col("source").cast(pl.String), pl.col("dest").cast(pl.String))
    .list.sort()
    .list.join("")
    .is_duplicated()
    .alias("duplicated")
)

In [ ]:
m = folium.Map(location=[50.949, 6.916], zoom_start=15)
fg_walking = folium.FeatureGroup("Walking")
# for node in df_nodes.iter_rows(named=True):
#     folium.Circle(
#         location=[node["lat"], node["long"]],
#         popup=f"Node {node['osm_id']}",
#         color="blue",
#         radius=1,
#     ).add_to(fg_walking)

for edge in df_edges.iter_rows(named=True):
    folium.PolyLine(
        locations=[
            [edge["from_lat"], edge["from_lon"]],
            [edge["dest_lat"], edge["dest_lon"]],
        ],
        color="blue" if not edge["duplicated"] else "lightblue",
        popup=f"{edge['source_osm']}",
        weight=2,
        opacity=1,
    ).add_to(fg_walking)
fg_walking.add_to(m)

fg_cycling = folium.FeatureGroup("Cycling")
# for node in df_c_nodes.iter_rows(named=True):
#     folium.Circle(
#         location=[node["lat"], node["long"]],
#         popup=f"Node {node['osm_id']}",
#         color="green",
#         radius=1,
#     ).add_to(fg_cycling)
for edge in df_c_edges.iter_rows(named=True):
    folium.PolyLine(
        locations=[
            [edge["from_lat"], edge["from_lon"]],
            [edge["dest_lat"], edge["dest_lon"]],
        ],
        color="green" if not edge["duplicated"] else "lightgreen",
        weight=2,
        opacity=1,
    ).add_to(fg_cycling)
fg_cycling.add_to(m)
fg_driving = folium.FeatureGroup("Driving")
# for node in df_d_nodes.iter_rows(named=True):
#     folium.Circle(
#         location=[node["lat"], node["long"]],
#         popup=f"Node {node['osm_id']}",
#         color="red",
#         radius=1,
#     ).add_to(fg_driving)
for edge in df_d_edges.iter_rows(named=True):
    folium.PolyLine(
        locations=[
            [edge["from_lat"], edge["from_lon"]],
            [edge["dest_lat"], edge["dest_lon"]],
        ],
        color="red" if not edge["duplicated"] else "lightcoral",
        weight=2,
        opacity=1,
    ).add_to(fg_driving)
fg_driving.add_to(m)


folium.LayerControl().add_to(m)
m

In [ ]:
m = folium.Map(location=[50.949, 6.916], zoom_start=15)
for node in df_pois.iter_rows(named=True):
    folium.Circle(
        location=[node["lat"], node["long"]],
        popup=f"Node {node['osm_id']}",
        radius=1,
        color="green",
    ).add_to(m)

    folium.Circle(
        location=[node["nearest_lat"], node["nearest_lon"]],
        popup=f"Node {node['nearest_osm_node']}",
        radius=1,
    ).add_to(m)

    folium.PolyLine(
        locations=[
            [node["lat"], node["long"]],
            [node["nearest_lat"], node["nearest_lon"]],
        ],
        color="red",
        weight=2,
        opacity=1,
    ).add_to(m)

for node in df_nodes.iter_rows(named=True):
    folium.Circle(
        location=[node["lat"], node["long"]],
        popup=f"Node {node['osm_id']}",
        radius=1,
    ).add_to(m)

m